# Validate Product Catalog
Validates raw product catalog table for further transformation.

In [0]:
SELECT * 
FROM ecommerce_catalog.bronze.product_catalog_raw;

In [0]:
DESCRIBE ecommerce_catalog.bronze.product_catalog_raw;

In [0]:
SELECT
  count(*) AS total_rows,
  count(DISTINCT product_id) AS unique_products,
  SUM(CASE WHEN active IS NULL THEN 1 ELSE 0 END) AS missing_active,
  SUM(CASE WHEN category IS NULL THEN 1 ELSE 0 END) AS missing_category,
  SUM(CASE WHEN current_list_price IS NULL THEN 1 ELSE 0 END) AS missing_current_list_price,
  SUM(CASE WHEN manufacturer IS NULL THEN 1 ELSE 0 END) AS missing_manufacturer,
  SUM(CASE WHEN product_id IS NULL THEN 1 ELSE 0 END) AS missing_product_id,
  SUM(CASE WHEN product_name IS NULL THEN 1 ELSE 0 END) AS missing_product_name,
  SUM(CASE WHEN stock_quantity IS NULL THEN 1 ELSE 0 END) AS missing_stock_quantity,
  SUM(CASE WHEN subcategory IS NULL THEN 1 ELSE 0 END) AS missing_subcategory,
  SUM(CASE WHEN supplier_country IS NULL THEN 1 ELSE 0 END) AS missing_supplier_country,
  SUM(CASE WHEN unit_cost IS NULL THEN 1 ELSE 0 END) AS missing_unit_cost,
  SUM(CASE WHEN process_date IS NULL THEN 1 ELSE 0 END) AS missing_process_date,
  SUM(CASE WHEN source_file IS NULL THEN 1 ELSE 0 END) AS missing_source_file
FROM ecommerce_catalog.bronze.product_catalog_raw;

In [0]:
SELECT active
FROM ecommerce_catalog.bronze.product_catalog_raw
ORDER BY active;






In [0]:
SELECT
  count(*) AS total_rows,
  SUM(CASE WHEN active = "TRUE" THEN 1 ELSE 0 END) AS status_TRUE,
  SUM(CASE WHEN active = "True" THEN 1 ELSE 0 END) AS status_True,
  SUM(CASE WHEN active = "true" THEN 1 ELSE 0 END) AS status_true,
  SUM(CASE WHEN active = "false" THEN 1 ELSE 0 END) AS status_false,
  SUM(CASE WHEN active = "FALSE" THEN 1 ELSE 0 END) AS status_FALSE,
  SUM(CASE WHEN active = "False" THEN 1 ELSE 0 END) AS status_False
FROM ecommerce_catalog.bronze.product_catalog_raw;


In [0]:
SELECT category
FROM ecommerce_catalog.bronze.product_catalog_raw
WHERE category = UPPER(category);

In [0]:
SELECT category
FROM ecommerce_catalog.bronze.product_catalog_raw
WHERE category = INITCAP(category);

In [0]:
SELECT category
FROM ecommerce_catalog.bronze.product_catalog_raw
WHERE category = lower(category);

In [0]:
SELECT category
FROM ecommerce_catalog.bronze.product_catalog_raw
WHERE category NOT LIKE "%&%"

In [0]:
SELECT current_list_price
FROM ecommerce_catalog.bronze.product_catalog_raw;

In [0]:
SELECT stock_quantity, product_id
FROM ecommerce_catalog.bronze.product_catalog_raw
ORDER BY stock_quantity DESC;

In [0]:
SELECT unit_cost, product_id
FROM ecommerce_catalog.bronze.product_catalog_raw
WHERE unit_cost < 0;

In [0]:
SELECT current_list_price, product_id
FROM ecommerce_catalog.bronze.product_catalog_raw
ORDER BY current_list_price;

### **Observations:**
- One `NULL` value was identified in `manufacturer`.
- One `NULL` value was identified in `subcategory`.
- Values in `active` exhibit inconsistent casing.
- Three inconsistent casings were identified in `category`: two entries in all uppercase and one entry in all lowercase.
- Two anomalous values were identified for `stock_quantity`:
    - `P0036` has a negative stock level of -5.
    - `P0104` has an exceptionally high stock level of 999999 compared to the remaining products.
- A negative `unit_cost` was identified for `P0022`.
- For certain products, product names and `subcategory` values do not logically align. For example, a watch is assigned to the `Shoes` subcategory, and a frying pan is assigned to `Bedding & Bath`. While syntactically valid, these represent semantic data quality issues.

### **Silver Layer Actions:**
- Standardize casing for `active` and `category`.
- Investigate missing values in `manufacturer` and `subcategory`, imputing them only where an unambiguous domain-level reconstruction is possible.
- Investigate anomalous values in `stock_quantity` and either clean them or treat them as non-recoverable, depending on available reference data.
- Investigate the negative `unit_cost` for `P0022`. A correction will only be applied if a plausible value can be derived from the available data.